# Zadanie 3.3: redukcja rzędu POD–Galerkin

Cel (issue #22): zbudować bazę POD (SVD migawek pola $u(x,t)$ z PDE) i zrzutować na nią
równanie Fishera–KPP z radioterapią metodą Galerkina (Holmes, Lumley, Berkooz 1996, rozdz. 3–4;
Rowley 2005, sekcje 2–3). Pytamy, **ile modów $k$ potrzeba**, żeby $M(t)$ i pole $u$ zgadzały się z PDE,
oraz jak baza zachowuje się na geometrii wiązki spoza zbioru uczącego.

$$u(x,t) \approx \sum_{j=1}^{k} a_j(t)\,\phi_j(x), \qquad
\dot a_i = \sum_j L_{ij} a_j - \tfrac{\rho}{K}\sum_{j,l} Q_{ijl} a_j a_l - \beta U(t) \sum_j B_{ij} a_j .$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid, initial_condition
from offtarget.observables import tumor_mass
from offtarget.pod import GalerkinDivergenceError, fit_pod, simulate_pod_galerkin
from offtarget.solvers import simulate

seed_everything()

x, dx = grid(80.0, 400)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
u0 = initial_condition(x)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)

plans = {
    "pole pokrywa guz, duża dawka": Params(beta=1.5, s=0.0, r=20.0),
    "pole pokrywa guz, mała dawka": Params(beta=0.45, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}
U_pde = {name: simulate(p, schedule, x, t, method="cn") for name, p in plans.items()}
M_pde = {name: tumor_mass(U, x) for name, U in U_pde.items()}

## 1. Baza POD: widmo wartości osobliwych i mody

Bazę budujemy wspólnie z migawek wszystkich czterech scenariuszy (4 × 201 migawek),
z ważonym iloczynem skalarnym $L^2$ (wagi trapezów), tak że $\Phi^\top W \Phi = I$.

In [ ]:
basis = fit_pod(list(U_pde.values()), x, max_modes=30)
energy = basis.cumulative_energy
k_needed = {thr: int(np.searchsorted(energy, thr) + 1) for thr in (0.99, 0.999, 0.9999)}
print("Liczba modów do progu energii:", k_needed)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
n_show = 30
axes[0].semilogy(np.arange(1, n_show + 1), basis.singular_values[:n_show], "o-", ms=4)
axes[0].set_xlabel("numer modu $i$")
axes[0].set_ylabel(r"wartość osobliwa $\sigma_i$")
axes[0].set_title("Widmo wartości osobliwych")
axes[0].grid(alpha=0.3, which="both")
ax2 = axes[0].twinx()
ax2.plot(np.arange(1, n_show + 1), 1.0 - energy[:n_show] + 1e-16, color="tab:red", ls="--")
ax2.set_yscale("log")
ax2.set_ylabel("niewyjaśniona energia $1 - E_k$", color="tab:red")

for i in range(5):
    axes[1].plot(x, basis.modes[:, i], lw=1.6, label=fr"$\phi_{i + 1}$")
axes[1].set_xlim(-30, 30)
axes[1].set_xlabel("położenie $x$")
axes[1].set_title("Pierwsze mody POD")
axes[1].legend(frameon=False, ncol=5, fontsize=9)
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_03_pod_galerkin_baza.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## 2. Ile modów potrzeba? Galerkin dla $k \in \{1, 2, 3, 5, 8, 12, 15, 20\}$

Porównujemy dwa warianty bazy:
- **baza własna** — POD z migawek tego samego scenariusza (klasyczne ustawienie redukcji rzędu),
- **baza wspólna** — POD z migawek wszystkich czterech scenariuszy naraz.

Dla każdego $k$ liczymy błąd względny $L^2$ masy $M(t)$ i pola $u(x,t)$ względem PDE oraz $\min \hat u$.
Obcięta baza daje oscylacje Gibbsa ($\hat u < 0$), które człon $-\rho u^2/K$ może napędzić do
$-\infty$ w skończonym czasie — wtedy zapisujemy „wybuch”.

In [ ]:
K_LIST = (1, 2, 3, 5, 8, 12, 15, 20)
own_basis = {name: fit_pod(U, x, max_modes=25) for name, U in U_pde.items()}
rows = []
traj = {}
for variant in ("własna", "wspólna"):
    for name, p in plans.items():
        b_used = own_basis[name] if variant == "własna" else basis
        for k in K_LIST:
            try:
                a, uh = simulate_pod_galerkin(u0, b_used, p, schedule, t, k=k)
                M_hat = b_used.truncate(k).mass(a)
                if variant == "własna":
                    traj[(name, k)] = M_hat
                rows.append({
                    "baza": variant, "scenariusz": name, "k": k, "status": "ok",
                    "blad_L2_M": round(float(np.linalg.norm(M_hat - M_pde[name]) / np.linalg.norm(M_pde[name])), 5),
                    "blad_L2_u": round(float(np.linalg.norm(uh - U_pde[name]) / np.linalg.norm(U_pde[name])), 5),
                    "min_u_hat": round(float(uh.min()), 4),
                })
            except GalerkinDivergenceError:
                rows.append({"baza": variant, "scenariusz": name, "k": k, "status": "wybuch",
                             "blad_L2_M": np.nan, "blad_L2_u": np.nan, "min_u_hat": np.nan})

df = pd.DataFrame(rows)
df.to_csv(TABLES / "zad3_03_pod_galerkin.csv", index=False)
for variant in ("własna", "wspólna"):
    sub = df[df["baza"] == variant]
    print(f"Baza {variant}: błąd względny L2 masy M(t) (NaN = wybuch)")
    print(sub.pivot(index="k", columns="scenariusz", values="blad_L2_M").to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharex=True, sharey=True)
for ax, variant in zip(axes, ("własna", "wspólna")):
    for name in plans:
        sub = df[(df["baza"] == variant) & (df["scenariusz"] == name) & (df["status"] == "ok")]
        ax.semilogy(sub["k"], sub["blad_L2_M"], "o-", label=name)
    ax.axhline(0.01, color="gray", ls=":", lw=1)
    ax.set_title(f"Baza {variant}: błąd względny $L^2$ masy $M(t)$")
    ax.set_xlabel("liczba modów $k$")
    ax.grid(alpha=0.3, which="both")
axes[0].legend(frameon=False, fontsize=8.5)
fig.suptitle("Zbieżność POD–Galerkin do PDE (brakujące punkty = wybuch numeryczny)", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "zad3_03_pod_galerkin_zbieznosc.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.5), sharex=True, sharey=True)
for ax, name in zip(axes.flat, plans):
    ax.plot(t, M_pde[name], color="black", lw=2.4, label="PDE (CN)")
    for k, col in zip((2, 5, 8, 15), ("tab:blue", "tab:purple", "tab:orange", "tab:green")):
        if (name, k) in traj:
            ax.plot(t, traj[(name, k)], color=col, lw=1.5, ls="--" if k < 8 else "-", label=f"POD–Galerkin, k={k}")
    for ti in schedule.times:
        ax.axvline(ti, color="tab:red", alpha=0.14, lw=0.8)
    ax.set_title(f"{name} (baza własna)")
    ax.grid(alpha=0.25)
axes[0, 0].legend(frameon=False, fontsize=8.5)
for ax in axes[1]:
    ax.set_xlabel("czas $t$")
for ax in axes[:, 0]:
    ax.set_ylabel("masa guza $M(t)$")
fig.tight_layout()
fig.savefig(FIGURES / "zad3_03_pod_galerkin_scenariusze.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## 3. Test poza zbiorem uczącym: nowa geometria wiązki

Baza z czterech scenariuszy, a ewaluacja na geometriach, których nie było w migawkach.

In [ ]:
ood = {
    "s=3, r=10 (interpolacja)": Params(beta=1.5, s=3.0, r=10.0),
    "s=10, r=6 (ekstrapolacja)": Params(beta=1.5, s=10.0, r=6.0),
    "s=0, r=5, beta=1.0": Params(beta=1.0, s=0.0, r=5.0),
}
ood_rows = []
for name, p in ood.items():
    U_ref = simulate(p, schedule, x, t, method="cn")
    M_ref = tumor_mass(U_ref, x)
    for k in (5, 8, 12, 15, 20):
        try:
            a, uh = simulate_pod_galerkin(u0, basis, p, schedule, t, k=k)
            M_hat = basis.truncate(k).mass(a)
            e_m = float(np.linalg.norm(M_hat - M_ref) / np.linalg.norm(M_ref))
            e_u = float(np.linalg.norm(uh - U_ref) / np.linalg.norm(U_ref))
            ood_rows.append({"geometria": name, "k": k, "status": "ok", "blad_L2_M": round(e_m, 5), "blad_L2_u": round(e_u, 5)})
        except GalerkinDivergenceError:
            ood_rows.append({"geometria": name, "k": k, "status": "wybuch", "blad_L2_M": np.nan, "blad_L2_u": np.nan})
    # rzut najlepszej możliwej aproksymacji w bazie (granica dla Galerkina)
    for k in (5, 8, 12, 15, 20):
        bk = basis.truncate(k)
        ood_rows.append({"geometria": name, "k": k, "status": "rzut (granica)",
                         "blad_L2_M": np.nan,
                         "blad_L2_u": round(float(np.linalg.norm(bk.reconstruct(bk.project(U_ref)) - U_ref) / np.linalg.norm(U_ref)), 5)})

df_ood = pd.DataFrame(ood_rows)
df_ood.to_csv(TABLES / "zad3_03_pod_galerkin_ood.csv", index=False)
print(df_ood.to_string(index=False))

## Wnioski

- **Energia a dokładność:** 99% energii migawek mieści się w 5 modach, 99,9% w 9, a 99,99% w 12.
  Przy bazie własnej scenariusza błąd $M(t)$ spada poniżej 1% dla $k \approx 8$–$12$, a dla $k \ge 15$
  jest rzędu $10^{-3}$ lub mniej we wszystkich czterech scenariuszach.
- **Zbieżność nie jest monotoniczna:** dla małych $k$ zdarzają się wybuchy (np. $k=3$ dla pola za wąskiego,
  $k=5$ dla pola przesuniętego). To znana wada zwykłej projekcji Galerkina dla nieliniowych PDE
  (Rowley 2005, sekcja 3): obcięta baza nie gwarantuje stabilności, a oscylacje $\hat u < 0$
  człon kwadratowy wzmacnia zamiast tłumić.
- **Baza wspólna i nowe geometrie:** baza z czterech scenariuszy jest wyraźnie mniej stabilna,
  a na geometriach wiązki spoza zbioru uczącego Galerkin najczęściej wybucha, mimo że sam rzut pola
  na tę bazę ma tylko 2–4% błędu przy $k=20$. Baza POD dobrze opisuje pole, ale model zredukowany
  nie przenosi się wiarygodnie na inne $R(x)$ bez stabilizacji (np. ograniczenia $\hat u \ge 0$,
  bazy z większej rodziny scenariuszy lub iloczynu skalarnego wzmacniającego stabilność).